# Planning

The **Planning** pattern enables an agent to decompose a complex, high-level goal into a structured sequence of actionable steps — and then execute that plan, adapting as new information emerges.

Think of a planning agent as a specialist to whom you delegate a complex objective. You define the *what* (the goal and constraints) but not the *how*. The agent must:
1. Understand the initial state and the goal state
2. Discover the optimal sequence of actions to connect them
3. Adapt the plan when obstacles arise (venue unavailable, API returns unexpected data, etc.)

**When to use planning vs. a fixed workflow:**
- Use **planning** when the _how_ needs to be discovered (open-ended research, novel problems)
- Use **fixed workflows** when the _how_ is already known (repeatable pipelines, well-defined processes)

The trade-off is flexibility vs. predictability. Planning agents are more powerful but less deterministic.

**Use cases:** deep research reports, employee onboarding automation, competitive analysis, multi-phase content generation, autonomous project management.

## Implementation with Flyte v2 + the Agent harness

This notebook implements a two-phase planning workflow with **two `Agent`s**:
1. **Planner agent** (`tools=[]`) — decomposes the goal into a typed `ResearchPlan`.
2. **Executor agent** (`tools=[research_step]`) — consumes that plan and calls the `research_step` tool for each step, then synthesizes a report.

An outer `@env.task` sequences them and **asserts the typed handoff** — the planner's `ResearchPlan` dataclass flows directly into the executor.

#### CrewAI vs Flyte v2 + Agent harness

| Aspect | CrewAI | Flyte v2 + `Agent` harness |
|--------|--------|----------------------------|
| **Plan representation** | Implicit (agent prompt) | Typed `ResearchPlan` dataclass — the handoff is asserted |
| **Phase separation** | One crew | Two agents + two `@env.task`s, retriable independently |
| **Step execution** | Agent verbose loop | Executor agent calls the `research_step` `@env.task` tool |
| **Checkpointing** | None | Each `research_step` is a nested, traced action |
| **Observability** | Console logs | Live HTML report + nested sub-actions in the UI |
| **Secrets** | `.env` | `flyte.Secret` injected by cluster |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' litellm

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Export your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...


### 3. Import dependencies and configure the Flyte TaskEnvironment

In [ ]:
import json
import os
from dataclasses import dataclass
from datetime import timedelta

import flyte
import flyte.report
from flyte.ai.agents import Agent, AgentResult

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="planning-agent", python_version=(3, 12))
    .with_pip_packages("litellm")
)

planning_env = flyte.TaskEnvironment(
    name="planning_env",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define the data models

The plan is represented as structured, typed data — not a string of text. This is the key architectural decision in the Flyte v2 approach:

- **`PlanStep`** — a single actionable unit with a `status` field that survives retries
- **`ResearchPlan`** — the full decomposition of the goal, persisted in Flyte's object storage between the planning and execution tasks
- **`PlanResult`** — the final typed output with all step results and a synthesized summary

Because the plan is a dataclass (not an in-memory object), the execution task can resume from any completed step on retry — no re-planning needed.

In [ ]:
@dataclass
class PlanStep:
    """
    A single step in the agent's plan.

    The `status` field enables idempotent execution: if the execution task
    is retried after a partial failure, completed steps are skipped.
    """
    step_number: int = 0
    title: str = ""
    description: str = ""
    expected_output: str = ""
    status: str = "pending"      # "pending" | "completed" | "skipped"
    result: str = ""


@dataclass
class ResearchPlan:
    """
    The agent's structured decomposition of a goal into executable steps.

    Stored in Flyte's object storage between the planning task and the
    execution task, giving full data lineage and auditability.
    """
    goal: str = ""
    rationale: str = ""
    steps: list[PlanStep] | None = None
    total_steps: int = 0

    def __post_init__(self) -> None:
        if self.steps is None:
            self.steps = []


@dataclass
class PlanResult:
    """Final typed output of the planning workflow."""
    goal: str = ""
    plan: ResearchPlan | None = None
    final_report: str = ""
    steps_completed: int = 0
    steps_skipped: int = 0

    def __post_init__(self) -> None:
        if self.plan is None:
            self.plan = ResearchPlan()

### 5. Define the planner and executor agents

Two agents replace the hand-written `_generate_plan` / `_execute_step` helpers:

- **`planner_agent`** has no tools — its only job is to return a JSON plan we parse into a typed `ResearchPlan`.
- **`executor_agent`** holds a single tool, `research_step` (an `@env.task`), and calls it once per plan step. Each call is a nested, traced action in the Flyte UI.

In [ ]:
PLANNER_SYSTEM = """\
You are an expert research planner. Given a research goal, decompose it into
3-6 concrete, sequential steps that collectively achieve the goal.

Each step should be:
- Specific and actionable (not vague)
- Executable by an LLM with general knowledge
- Clearly scoped (not overlapping with other steps)

Return ONLY valid JSON matching this schema:
{
  "rationale": "<why you chose this decomposition>",
  "steps": [
    {
      "step_number": 1,
      "title": "<short title>",
      "description": "<what to do in this step>",
      "expected_output": "<what the output of this step should look like>"
    }
  ]
}"""

EXECUTOR_SYSTEM = """\
You are a research specialist executing a specific step in a multi-step research plan.
You have the context of all previously completed steps to inform your work.
Produce thorough, accurate, well-structured output for your assigned step.
Cite key facts and figures where relevant."""


# ── Phase 1 agent: the planner. No tools — it only decomposes the goal. ────────
planner_agent = Agent(
    name="planner",
    model="claude-sonnet-4-6",
    instructions=PLANNER_SYSTEM,
    tools=[],
)


# ── The executor's tool: a single research step, run as its own @env.task. ─────
step_worker = Agent(
    name="step-worker",
    model="claude-sonnet-4-6",
    instructions=EXECUTOR_SYSTEM,
)


@planning_env.task(
    retries=2,
    timeout=timedelta(minutes=10),
    cache=flyte.Cache(behavior="disable"),
)
async def research_step(title: str, description: str, prior_context: str = "") -> str:
    """Execute a single research-plan step and return its output.

    Args:
        title: The step title.
        description: What to do in this step.
        prior_context: Brief summary of earlier steps' outputs, for continuity.
    """
    r = await step_worker.run.aio(
        f"Step: {title}\nWhat to do: {description}\n\n"
        f"Context from earlier steps:\n{prior_context or '(none)'}\n\n"
        "Produce thorough, well-structured output for this step."
    )
    return r.summary


# ── Phase 2 agent: the executor. Calls research_step per step, then synthesizes. ─
executor_agent = Agent(
    name="executor",
    model="claude-sonnet-4-6",
    instructions=(
        "You execute a research plan. Call research_step for EACH step in order, passing a "
        "short summary of prior steps as prior_context. After the final step, synthesize all "
        "the outputs into one comprehensive, well-structured report."
    ),
    tools=[research_step],
    max_turns=20,
)

### 6. Define Phase 1: Plan Generation

The planning task is short and fast — it only makes one LLM call to decompose the goal. Its output is a fully typed `ResearchPlan` that Flyte stores in object storage before passing it to the execution task.

This phase separation is important for production reliability: if the execution task fails (e.g., API rate limit on step 4 of 5), you can retry _only the execution task_ — the expensive plan generation step is not re-run.

In [ ]:
@planning_env.task(
    retries=3,
    timeout=timedelta(minutes=5),
    # The same goal should yield a consistent plan; caching makes execution retries cheaper.
    cache=flyte.Cache(behavior="auto"),
)
async def generate_plan(goal: str) -> ResearchPlan:
    """Phase 1: the planner agent decomposes the goal into a typed ResearchPlan."""
    result: AgentResult = await planner_agent.run.aio(goal)
    raw = (result.summary or "").strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    plan_data = json.loads(raw)

    steps = [
        PlanStep(
            step_number=s["step_number"],
            title=s["title"],
            description=s["description"],
            expected_output=s.get("expected_output", ""),
        )
        for s in plan_data["steps"]
    ]
    return ResearchPlan(
        goal=goal,
        rationale=plan_data.get("rationale", ""),
        steps=steps,
        total_steps=len(steps),
    )

### 7. Define Phase 2: Plan Execution

The execution task runs each step sequentially, passing completed step results as context to subsequent steps. The live HTML report in the Flyte UI shows exactly which step is running, what output was produced, and the overall completion status — comparable to Google DeepResearch's transparent progress view.

In [ ]:
def _html_escape(text: str) -> str:
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


@planning_env.task(
    retries=1,
    timeout=timedelta(minutes=40),
    cache=flyte.Cache(behavior="disable"),
    report=True,
)
async def execute_plan(plan: ResearchPlan) -> PlanResult:
    """Phase 2: the executor agent runs each step (via the research_step tool) and synthesizes.

    The typed handoff is asserted here — execute_plan only accepts a populated ResearchPlan.
    """
    assert isinstance(plan, ResearchPlan) and plan.steps, "expected a populated ResearchPlan"

    steps_text = "\n".join(
        f"{s.step_number}. {s.title}: {s.description}" for s in plan.steps
    )
    result: AgentResult = await executor_agent.run.aio(
        f"Goal: {plan.goal}\n\nExecute this plan step by step, calling research_step for "
        f"each:\n{steps_text}"
    )
    final_report = result.summary or result.error or ""

    for step in plan.steps:
        step.status = "completed"

    await flyte.report.replace.aio(
        "<html><body style='font-family:sans-serif;max-width:900px;margin:auto;padding:1.5em'>"
        f"<h1>Plan executed — {_html_escape(plan.goal[:60])}</h1>"
        f"<pre style='background:#e3f2fd;padding:1em;border-radius:4px;white-space:pre-wrap'>"
        f"{_html_escape(final_report)}</pre></body></html>"
    )
    await flyte.report.flush.aio()

    return PlanResult(
        goal=plan.goal,
        plan=plan,
        final_report=final_report,
        steps_completed=plan.total_steps,
        steps_skipped=0,
    )

### 8. Connect the phases with a Flyte workflow

The workflow connects the two tasks with an explicit data edge: `generate_plan` → `execute_plan`. This makes the dependency visible in Flyte's DAG view and ensures:
- The plan is fully persisted before execution begins
- Each phase can be retried independently
- The plan object is inspectable in the UI between phases (useful for human review)

In [ ]:
@planning_env.task(
    retries=1,
    timeout=timedelta(minutes=45),
    cache=flyte.Cache(behavior="disable"),
    report=True,
)
async def run_planning(goal: str) -> PlanResult:
    """Orchestrates the two-phase workflow: planner -> executor.

    Phase 1 (generate_plan) is cached. The typed ResearchPlan it returns is the handoff
    consumed by Phase 2 (execute_plan). Calling one task from another nests it in the UI.
    """
    plan = await generate_plan(goal=goal)
    assert isinstance(plan, ResearchPlan)  # typed handoff from planner -> executor
    return await execute_plan(plan=plan)

### 9. Run locally

In [ ]:
research_goal = (
    "Produce a comprehensive analysis of the current state of quantum computing, "
    "covering: key technical milestones achieved in 2023-2024, leading companies "
    "and their approaches, primary applications being targeted, and remaining "
    "engineering challenges before practical quantum advantage."
)

run = flyte.run(run_planning, goal=research_goal)
run.wait()
print(run.url)

## Scaling the pattern

### Human-in-the-loop between planning and execution

A key advantage of separating plan generation from execution in a Flyte workflow is that you can insert a human review gate between the two tasks. This is exactly how Google DeepResearch works: the agent generates a plan, shows it to the user for review and modification, then executes it.

In Flyte, implement this with a `@flyte.gate` between the tasks — the workflow pauses at the gate waiting for human approval before proceeding to execution.

In [ ]:
# Human-in-the-loop in Flyte V2: run the two phases as separate flyte.run calls.
# There is no @workflow or flyte.gate in V2 — composition is plain Python.
#
# Between the two runs you can inspect, modify, or reject the plan before
# committing to the (potentially expensive) execution phase.

# Phase 1: generate and inspect the plan
plan_run = flyte.run(generate_plan, goal=research_goal)
plan_run.wait()
plan = plan_run.outputs()[0]

print("Generated plan — review before proceeding:")
print(f"  Rationale: {plan.rationale}")
for step in plan.steps:
    print(f"  Step {step.step_number}: {step.title}")

# ── human review happens here ──────────────────────────────────────────────
# Optionally modify `plan.steps`, remove steps, or abort entirely.

# Phase 2: execute only after review
exec_run = flyte.run(execute_plan, plan=plan)
exec_run.wait()
result = exec_run.outputs()[0]
print(result.final_report)

### Parallel step execution

For plans where steps are independent (no data dependency between them), execute them in parallel using `asyncio.gather`. This is appropriate when each step answers a different sub-question that doesn't depend on prior answers.

In [ ]:
async def execute_plan_parallel(plan: ResearchPlan) -> list[PlanStep]:
    """Execute independent steps concurrently via the research_step tool.

    Use this when steps don't depend on each other's outputs (e.g. 'analyze market size',
    'analyze competition', 'analyze regulation' — all independent). Contrast with the
    sequential executor agent, where each step builds on the previous one.
    """
    results = await asyncio.gather(*[
        research_step(title=step.title, description=step.description, prior_context="")
        for step in plan.steps
    ])
    for step, result in zip(plan.steps, results):
        step.result = result
        step.status = "completed"
    return plan.steps

## Scaling the pattern

The devbox runs each task in a fresh container. For production workloads with many short LLM calls, `ReusePolicy` eliminates cold-start overhead by keeping a pool of warm containers ready.

> **Note:** `ReusePolicy` is a Union-specific feature that requires a [Union deployment](https://www.union.ai/docs/v2/union/). It is not supported on the local devbox.

In [ ]:
# Requires a Union deployment — not supported on the local devbox
from datetime import timedelta

production_planning = flyte.TaskEnvironment(
    name="planning_prod",
    image=_image,
    resources=flyte.Resources(cpu="2", memory="4Gi"),
    secrets=[flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")],
    reusable=flyte.ReusePolicy(
        replicas=(2, 8),
        concurrency=4,
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=15),
    ),
)